In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.utils import resample
from scipy.stats import mode

In [20]:
df = pd.read_csv("titanic_prepared.csv", index_col=0)

Y = df['label']
X = df.drop('label', axis=1)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.1, random_state=42, stratify=Y
)

In [ ]:
class MyRandomForest:
    def __init__(self, n_estimators=100, max_features='sqrt',
                 max_depth=None, min_samples_split=2, random_state=None):
        # Количество деревьев в лесу
        self.n_estimators = n_estimators
        # Сколько признаков использовать для каждого дерева ('sqrt', 'log2' или число)
        self.max_features = max_features
        # Максимальная глубина дерева
        self.max_depth = max_depth
        # Минимальное число объектов в узле для дальнейшего деления
        self.min_samples_split = min_samples_split
        # random_state — для воспроизводимости результатов
        self.random_state = random_state

        # Список всех обученных деревьев
        self.trees = []
        # Список списков индексов признаков, использованных каждым деревом
        self.feature_indices = []


    def fit(self, X, Y):
        # Если данные переданы как DataFrame — переводим в numpy массив
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(Y, pd.DataFrame):
            Y = Y.values

        # Перед обучением очищаем список деревьев
        self.trees = []
        self.feature_indices = []

        # Фиксируем numpy seed для повторяемости
        np.random.seed(self.random_state)

        # n_sample — кол-во объектов, n_features — кол-во признаков
        n_sample, n_features = X.shape

        # Определяем, сколько признаков использовать в каждом дереве
        if self.max_features == 'sqrt':
            max_features = int(np.sqrt(n_features))  # классика для классификации
        elif self.max_features == 'log2':
            max_features = int(np.log2(n_features))
        elif isinstance(self.max_features, (int, float)):
            max_features = int(self.max_features)
        else:
            max_features = n_features  # все признаки

        # Генератор случайных чисел, завязанный на random_state
        if self.random_state is not None:
            rng = np.random.RandomState(self.random_state)
        else:
            rng = np.random

        # Строим n_estimators деревьев
        for _ in range(self.n_estimators):

            # Уникальный seed для каждого дерева
            tree_random_state = (
                rng.randint(0, 2**31 - 1) if self.random_state is not None else None
            )

            # Делаем бутстрэп: случайная выборка с возвращением
            X_sample, Y_sample = resample(
                X, Y,
                random_state=tree_random_state,  # у каждого дерева свой seed
                n_samples=n_sample
            )

            # Случайно выбираем подмножество признаков
            feature_idx = np.random.choice(n_features, max_features, replace=False)
            # Берём данные только по выбранным признакам
            X_subset = X_sample[:, feature_idx]

            # Создаём дерево решений
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                random_state=tree_random_state,
                max_features=None   # признаки уже выбраны вручную
            )

            # Обучаем дерево
            tree.fit(X_subset, Y_sample)

            # Сохраняем дерево и список использованных признаков
            self.trees.append(tree)
            self.feature_indices.append(feature_idx)

        return self  # для совместимости со sklearn API


    def predict(self, X):
        # Переводим DataFrame в numpy, если нужно
        if isinstance(X, pd.DataFrame):
            X = X.values

        # Матрица предсказаний всех деревьев
        predictions = np.zeros((X.shape[0], self.n_estimators))

        # Сбор предсказаний от каждого дерева
        for i, (tree, feature_idx) in enumerate(zip(self.trees, self.feature_indices)):
            # Берём те же признаки, что дерево использовало на обучении
            X_subset = X[:, feature_idx]
            # Заполняем i-й столбец предсказаний
            predictions[:, i] = tree.predict(X_subset)

        # Берём моду (самое частое значение по строке)
        y_pred = mode(predictions, axis=1)[0]
        return y_pred.ravel()  # превращаем в 1D массив


    def score(self, X, y):
        # Стандартная метрика accuracy = доля верных предсказаний
        return accuracy_score(y, self.predict(X))


    def get_params(self, deep=True):
        # Возвращаем параметры модели — нужно для GridSearchCV
        return {
            'n_estimators': self.n_estimators,
            'max_features': self.max_features,
            'max_depth': self.max_depth,
            'min_samples_split': self.min_samples_split,
            'random_state': self.random_state
        }

    def set_params(self, **params):
        # Установка параметров модели — тоже для совместимости со sklearn
        for k, v in params.items():
            setattr(self, k, v)
        return self


In [22]:
# --- XGBoost ---
XGBoost_parameters = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.05, 0.1],
}
XGBoost_Grid = GridSearchCV(
    XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss'),
    XGBoost_parameters,
    cv=5,
    scoring='accuracy'
)
XGBoost_Grid.fit(X_train, Y_train)
XGBoost_best = XGBoost_Grid.best_estimator_

# --- Logistic Regression ---
logr_parameters = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    'solver': ['liblinear', 'lbfgs']
}
logr_Grid = GridSearchCV(
    LogisticRegression(max_iter=10000, random_state=42),
    logr_parameters,
    cv=5,
    scoring='accuracy'
)
logr_Grid.fit(X_train, Y_train)
logr_best = logr_Grid.best_estimator_

# --- Decision Tree ---
dTree_parameters = {
    'max_depth': np.arange(2, 21),
    'criterion': ['gini', 'entropy']
}
dTree_Grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dTree_parameters,
    cv=5,
    scoring='accuracy'
)
dTree_Grid.fit(X_train, Y_train)
dTree_best = dTree_Grid.best_estimator_


In [23]:
models = {
    'XGBoost': XGBoost_best,
    'LogReg': logr_best,
    'DecisionTree': dTree_best
}

print("\nТОЧНОСТЬ МОДЕЛЕЙ НА TEST:")
for name, model in models.items():
    model.fit(X_train, Y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(Y_test, pred)
    print(f"{name:15s}: {acc:.4f} {'OK' if acc >= 0.85 else 'LOW'}")


ТОЧНОСТЬ МОДЕЛЕЙ НА TEST:
XGBoost        : 0.8916 OK
LogReg         : 0.8584 OK
DecisionTree   : 0.8931 OK


In [24]:
dTree_best.fit(X_train, Y_train)

importances = dTree_best.feature_importances_
indices = np.argsort(importances)
features = X_train.columns

top2_idx = indices[-2:]
top2_features = features[top2_idx]

print("\nДВА САМЫХ ВАЖНЫХ ПРИЗНАКА:")
for f in top2_features:
    print("  -", f)

X_train_top = X_train[top2_features]
X_test_top = X_test[top2_features]

dt_top2 = DecisionTreeClassifier(
    max_depth=dTree_best.max_depth,
    criterion=dTree_best.criterion,
    random_state=42
)
dt_top2.fit(X_train_top, Y_train)

acc_top2 = accuracy_score(Y_test, dt_top2.predict(X_test_top))

print(f"\nAccuracy дерева на 2 признаках: {acc_top2:.4f}")


ДВА САМЫХ ВАЖНЫХ ПРИЗНАКА:
  - day
  - morning

Accuracy дерева на 2 признаках: 0.8614


In [ ]:
print("\nСРАВНЕНИЕ: ОДНО ДЕРЕВО VS MyRandomForest")

single_tree = DecisionTreeClassifier(max_depth=4, min_samples_split=2, random_state=42)
single_tree.fit(X_train, Y_train)
acc_single = accuracy_score(Y_test, single_tree.predict(X_test))

rf = MyRandomForest(n_estimators=200, max_features='sqrt', random_state=42)
rf.fit(X_train, Y_train)
acc_rf = accuracy_score(Y_test, rf.predict(X_test))

print(f"Одно дерево:     {acc_single:.4f}")
print(f"RandomForest: {acc_rf:.4f}")
print(f"Разница:         {acc_rf - acc_single:.4f}")


СРАВНЕНИЕ: ОДНО ДЕРЕВО VS MyRandomForest
Одно дерево:     0.8855
Мой RandomForest: 0.8464
Разница:         -0.0392
